# 🧠 Custom Training Template: Text-Guided Occluded Image Completion
This notebook provides a training pipeline to fine-tune Stable Diffusion for inpainting with user-defined text prompts and masked images.

In [ ]:
!pip install diffusers transformers accelerate datasets safetensors

## 📁 Dataset Format
```
/dataset
    /images
        img001.png
        img002.png
    /masks
        img001_mask.png
        img002_mask.png
    prompts.txt
        img001.png|A red car parked under a tree.
        img002.png|A dog sitting in the middle of a field.
```

In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import torch

class InpaintingDataset(Dataset):
    def __init__(self, image_dir, mask_dir, prompt_file, transform):
        self.entries = []
        with open(prompt_file) as f:
            for line in f:
                img, prompt = line.strip().split("|")
                self.entries.append((img, prompt))
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        img_name, prompt = self.entries[idx]
        image = Image.open(f"{self.image_dir}/{img_name}").convert("RGB").resize((512, 512))
        mask = Image.open(f"{self.mask_dir}/{img_name.replace('.png', '_mask.png')}").convert("L").resize((512, 512))
        image = self.transform(image)
        mask = self.transform(mask)
        return image, mask, prompt

## 🔁 Training Loop (Simplified Pseudo-Code)

In [ ]:
# for image, mask, prompt in dataloader:
#     text_embeddings = text_encoder(prompt)
#     noised_image = forward_diffusion(image)
#     predicted_noise = unet(noised_image, mask, text_embeddings)
#     loss = MSE(predicted_noise, target_noise)
#     loss.backward()
#     optimizer.step()

### 📌 Notes:
- Freeze VAE and text encoder.
- Fine-tune only the UNet using masked image + text pairs.
- Use LoRA or DreamBooth if working with a small dataset.
- Add CLIP loss optionally for stronger text-image alignment.